In [1]:
# pip install pandas pyarrow lightgbm scikit-learn matplotlib

In [47]:
import pandas as pd
import numpy as np

df = pd.read_parquet("C:\Desktop\hft\data\ml_features_1m_v2.parquet")
df = df.sort_values(["market_id", "minute_bar"]).reset_index(drop=True)

H = 5  # horizon

# forward return over next H minutes
fwd = df.groupby("market_id")["close_mid"].shift(-H) / df["close_mid"] - 1

# keep only rows that are multiples of H (non-overlapping)
df["idx_in_market"] = df.groupby("market_id").cumcount()
mask = (df["idx_in_market"] % H == 0)

df = df[mask].copy()
df["target"] = (fwd[mask] > 0).astype(int)

In [48]:
# rolling (past only)
df["ofi_ma_5"] = df.groupby("market_id")["order_flow_imbalance"]\
    .rolling(5).mean().reset_index(level=0, drop=True)

df["volatility_ma_5"] = df.groupby("market_id")["bar_volatility"]\
    .rolling(5).mean().reset_index(level=0, drop=True)

df["price_momentum"] = df.groupby("market_id")["close_mid"].pct_change(3)

df["depth_pressure"] = df["bid_depth"] - df["ask_depth"]

df["ofi_vol"] = df["order_flow_imbalance"] * df["bar_volatility"]
df["spread_vol_ratio"] = df["mean_spread"] / (df["bar_volatility"] + 1e-6)

In [49]:
feature_cols = [
    "order_flow_imbalance",
    "bar_volatility",
    "mean_spread",
    "trade_count",
    "depth_imbalance",
    "bid_depth",
    "ask_depth",
    "ofi_ma_5",
    "volatility_ma_5",
    "price_momentum",
    "depth_pressure",
    "ofi_vol",
    "spread_vol_ratio"
]

In [50]:
df[feature_cols] = df.groupby("market_id")[feature_cols].shift(1)
df = df.dropna().reset_index(drop=True)

In [51]:
train_parts = []
test_parts = []

for mkt, g in df.groupby("market_id"):
    split = int(len(g) * 0.7)
    train_parts.append(g.iloc[:split])
    test_parts.append(g.iloc[split:])

train_df = pd.concat(train_parts)
test_df = pd.concat(test_parts)

In [52]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

X_train = train_df[feature_cols]
y_train = train_df["target"]

X_test = test_df[feature_cols]
y_test = test_df["target"]

model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6
)

model.fit(X_train, y_train)

y_pred = model.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, y_pred))

[LightGBM] [Info] Number of positive: 120105, number of negative: 644896
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018851 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3087
[LightGBM] [Info] Number of data points in the train set: 765001, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.157000 -> initscore=-1.680723
[LightGBM] [Info] Start training from score -1.680723
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
AUC: 0.8006197981170872
